# Single-conv readout ablation
Compare channel-max kernel response with a learned linear readout. Both runs keep LGT, one Conv1D(512,128,k=5), Top-K video pooling, identical seed and training settings. No existing checkpoints are overwritten.

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
FEATURE_ROOT = Path('/content/UCFClipFeatures')
RESULT_ROOT = PROJECT_ROOT / 'code' / 'results'
RUN_TAG = 'cnn_single_readout_ablation_s234'
VARIANTS = {'kernel_max': 'kernel_response', 'linear_readout': 'scoring_head'}

if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime first.')
if not SRC_DIR.is_dir():
    raise FileNotFoundError(SRC_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'scipy', 'scikit-learn', 'pandas'], check=True)
if not FEATURE_ROOT.is_dir():
    archives = [PROJECT_ROOT / name for name in ('UCFClipFeatures.zip', 'UCFClipFeatures.tar', 'UCFClipFeatures.tar.gz')]
    archive = next((path for path in archives if path.is_file()), None)
    if archive is None:
        raise FileNotFoundError('Missing /content/UCFClipFeatures and feature archive on Drive.')
    local_archive = Path('/content') / archive.name
    if not local_archive.is_file() or local_archive.stat().st_size != archive.stat().st_size:
        shutil.copy2(archive, local_archive)
    command = ['unzip', '-q', str(local_archive), '-d', '/content'] if archive.suffix == '.zip' else ['tar', '-xf', str(local_archive), '-C', '/content']
    subprocess.run(command, check=True)
for required in (FEATURE_ROOT, SRC_DIR / 'model_cnn_vadclip.py', SRC_DIR / 'ucf_train_cnn_vadclip.py',
                 SRC_DIR / 'ucf_evaluate_cnn_vadclip.py', SRC_DIR / 'test_cnn_single_conv.py',
                 LIST_DIR / 'ucf_CLIP_rgb_relative.csv', LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv',
                 LIST_DIR / 'gt_ucf.npy', LIST_DIR / 'gt_segment_ucf.npy', LIST_DIR / 'gt_label_ucf.npy'):
    if not required.exists():
        raise FileNotFoundError(required)
print('GPU:', torch.cuda.get_device_name(0))
print('Feature root:', FEATURE_ROOT)
print('Run tag:', RUN_TAG)


In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'test_cnn_single_conv'], cwd=SRC_DIR, check=True)

def run_command(command):
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, cwd=SRC_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise RuntimeError('Child process failed; traceback is printed above.')

for name, mode in VARIANTS.items():
    print('\n' + '=' * 70 + '\nVariant:', name, flush=True)
    epoch_dir = SRC_DIR / 'model' / 'epoch_checkpoints_cnn_single_conv' / RUN_TAG / name
    model_path = epoch_dir / 'epoch_010.pth'
    train_dir = RESULT_ROOT / 'train' / 'cnn_single_conv' / RUN_TAG / name
    eval_dir = RESULT_ROOT / 'evaluate' / 'cnn_single_conv' / RUN_TAG / name
    analyze_dir = RESULT_ROOT / 'analyze' / 'cnn_single_conv' / RUN_TAG / name
    config_path = train_dir / 'config.json'
    if model_path.is_file():
        if not config_path.is_file():
            raise RuntimeError(f'Checkpoint without config: {model_path}. Change RUN_TAG.')
        config = json.loads(config_path.read_text(encoding='utf-8'))
        expected = {'cnn_architecture': 'single_conv', 'score_mode': mode, 'aggregation': 'topk',
                    'kernel_size': 5, 'cnn_channels': 128, 'seed': 234, 'batch_size': 64,
                    'lr': 1e-5, 'max_epoch': 10, 'scheduler_milestones': [4, 8],
                    'lambda_alignment': 1.0, 'lambda_text': 1.0,
                    'feature_root': str(FEATURE_ROOT),
                    'train_list': str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv')}
        if any(config.get(key) != value for key, value in expected.items()):
            raise RuntimeError(f'Checkpoint config mismatch: {model_path}. Change RUN_TAG.')
        print('Matching checkpoint found; skip training:', model_path)
    else:
        run_command([sys.executable, '-u', str(SRC_DIR / 'ucf_train_cnn_vadclip.py'),
                     '--feature-root', str(FEATURE_ROOT),
                     '--train-list', str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv'),
                     '--cnn-architecture', 'single_conv', '--score-mode', mode,
                     '--aggregation', 'topk', '--kernel-size', '5', '--cnn-channels', '128',
                     '--lambda-alignment', '1', '--lambda-text', '1',
                     '--seed', '234', '--batch-size', '64', '--lr', '1e-5', '--max-epoch', '10',
                     '--scheduler-milestones', '4', '8', '--num-workers', '2',
                     '--epoch-checkpoint-dir', str(epoch_dir),
                     '--output-model-path', str(SRC_DIR / 'model' / 'cnn_single_conv' / RUN_TAG / f'{name}.pth'),
                     '--results-dir', str(train_dir)])
    metrics_path = eval_dir / 'metrics.json'
    if metrics_path.is_file():
        cached = json.loads(metrics_path.read_text(encoding='utf-8'))
        if cached.get('model_path') != str(model_path) or cached.get('score_mode') != mode:
            raise RuntimeError(f'Cached metrics mismatch: {metrics_path}. Change RUN_TAG.')
        print('Matching evaluation found; skip:', metrics_path)
    else:
        run_command([sys.executable, '-u', str(SRC_DIR / 'ucf_evaluate_cnn_vadclip.py'),
                     '--model-type', 'cnn', '--model-path', str(model_path),
                     '--cnn-architecture', 'single_conv', '--score-mode', mode,
                     '--kernel-size', '5', '--cnn-channels', '128',
                     '--train-config-json', str(config_path),
                     '--feature-root', str(FEATURE_ROOT),
                     '--test-list', str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
                     '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
                     '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
                     '--gt-label-path', str(LIST_DIR / 'gt_label_ucf.npy'),
                     '--evaluate-dir', str(eval_dir), '--analyze-dir', str(analyze_dir)])
    metric = json.loads(metrics_path.read_text(encoding='utf-8'))
    print(name, 'AUC1', round(metric['classifier_auc'], 2),
          'AP1', round(metric['classifier_ap'], 2),
          'AUC2', round(metric['alignment_auc'], 2),
          'mAP', round(metric['avg_mAP'], 2))


In [ ]:
rows = []
for name in VARIANTS:
    path = RESULT_ROOT / 'evaluate' / 'cnn_single_conv' / RUN_TAG / name / 'metrics.json'
    metric = json.loads(path.read_text(encoding='utf-8'))
    rows.append((name, metric))
print('Variant | AUC1 | AP1 | AUC2 | AP2 | Ano-AUC | mAP | Top-15 macro')
for name, metric in rows:
    summary_path = RESULT_ROOT / 'analyze' / 'cnn_single_conv' / RUN_TAG / name / 'summary.json'
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    top15 = summary['selected_anomaly_fraction_abnormal_video_macro'] * 100
    print(f"{name} | {metric['classifier_auc']:.2f} | {metric['classifier_ap']:.2f} | "
          f"{metric['alignment_auc']:.2f} | {metric['alignment_ap']:.2f} | "
          f"{metric['ano_auc']:.2f} | {metric['avg_mAP']:.2f} | {top15:.2f}%")
    groups = summary['groups']
    print('  Score p50: true-high', round(groups['true_high_abnormal']['score_p50'], 3),
          'false-high (abnormal video)', round(groups['false_high_abnormal']['score_p50'], 3),
          'false-high (normal video)', round(groups['false_high_normal']['score_p50'], 3))
    print('  Missed-anomaly score p10:', round(groups['missed_anomaly']['score_p10'], 3))
a, b = [metric for _, metric in rows]
print('Linear minus kernel-max AUC1:', round(b['classifier_auc'] - a['classifier_auc'], 2))
print('Output root:', RESULT_ROOT / 'evaluate' / 'cnn_single_conv' / RUN_TAG)
